In [51]:
import sys
sys.path.append('..')
from sqlalchemy import create_engine, text
import os

import geopandas as gpd
import pandas as pd
from geoalchemy2 import Geometry

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

POSTGRES_UTEA['DATABASE'] = 'utea_precision'

In [52]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

def obtener_lotes_para_segmentar():
    engine = obtener_engine()
    
    # Filtramos por lineas_creadas = True y segmentos_creados = False
    query = text("""
        SELECT * FROM siembra_surcado.data_lote 
        WHERE segmentos_creados = :segmentos
    """)
    
    try:
        with engine.connect() as conn:
            df = pd.read_sql(
                query, 
                conn, 
                params={
                    "segmentos": False
                }
            )
        return df
    except Exception as e:
        print(f"Error al obtener lotes para segmentar: {e}")
        return None
    
def limpiar_segmentos_por_distancia_velocidad(lote_id):
    """
    Elimina registros donde el segmento mide más de 2 unidades 
    y la velocidad del vehículo es 0.
    """
    engine = obtener_engine()
    
    # ST_Length calcula la longitud intrínseca de la geometría en esa fila
    query_delete = text("""
        DELETE FROM siembra_surcado.segmentos_lineas
        WHERE data_lote_id = :lote_id
          AND (ST_Length(geom) > 2
          OR vehiclspee = 0)
    """)
    
    try:
        with engine.begin() as conn:
            result = conn.execute(query_delete, {"lote_id": lote_id})
            print(f"Limpieza completada. Segmentos eliminados: {result.rowcount}")
            return True
    except Exception as e:
        print(f"Error al ejecutar la limpieza: {e}")
        return False

def limpiar_segementos_por_rumbo(lote_id, tolerancia=5):
    """
    Calcula los rumbos predominantes y elimina los registros 
    que no pertenecen al eje de surcado directamente en la tabla.
    """
    engine = obtener_engine()
    
    query = text("""
        WITH rumbos_stats AS (
            -- 1. Encontramos el rumbo más frecuente (redondeado)
            SELECT ROUND(heading)::numeric as rumbo_principal
            FROM siembra_surcado.segmentos_lineas
            WHERE data_lote_id = :lote_id
            GROUP BY ROUND(heading)
            ORDER BY COUNT(*) DESC
            LIMIT 1
        ),
        ejes AS (
            -- 2. Definimos el eje de ida y el de vuelta (180°) usando el operador %
            SELECT 
                rumbo_principal as ida,
                (rumbo_principal + 180) % 360 as vuelta
            FROM rumbos_stats
        )
        DELETE FROM siembra_surcado.segmentos_lineas
        WHERE data_lote_id = :lote_id
          AND id NOT IN (
            SELECT s.id 
            FROM siembra_surcado.segmentos_lineas s, ejes e
            WHERE 
                -- Filtro para rumbo de ida con manejo de circularidad
                ABS(((s.heading - e.ida + 540)::numeric % 360) - 180) <= :tol
                OR
                -- Filtro para rumbo de vuelta con manejo de circularidad
                ABS(((s.heading - e.vuelta + 540)::numeric % 360) - 180) <= :tol
          );
    """)
    
    try:
        with engine.begin() as conn:
            result = conn.execute(query, {"lote_id": lote_id, "tol": tolerancia})
            print(f"Limpieza de rumbos completada. Registros eliminados: {result.rowcount}")
            return True
    except Exception as e:
        print(f"Error al limpiar rumbos en DB: {e}")
        return False

In [69]:
# Uso
lotes_a_segmentar = obtener_lotes_para_segmentar()

In [70]:
lotes_a_segmentar

,id,unidad_01,unidad_02,unidad_05,campanha,fecha_de_registro,puntos_cargados,lineas_creadas,segmentos_creados,desviacion_calculada,velocidad_calculada,lote_ref,prop_ref,plano_generado,vehiclspee_calculada
0,1,2238,EL PARAISO,L5,2026,2026-03-05,True,False,False,False,False,AB L5,PARAISO,False,True


In [73]:
def procesar_segmentacion_exacta():
    engine = obtener_engine()
    
    # 1. Recorrer los IDs de data_lote (lineas_creadas=True, segmentos_creados=False)
    query_lotes = text("""
        SELECT id FROM siembra_surcado.data_lote 
        WHERE segmentos_creados = False
    """)
    
    try:
        with engine.begin() as conn:
            lotes = conn.execute(query_lotes).fetchall()
            for (lote_id,) in lotes:
                print(f"--- Procesando Lote ID: {lote_id} ---")
                # 3. Seleccionar puntos por INTERSECCIÓN con la línea y filtrar por lote_id
                # Ordenamos por isotime para garantizar la secuencia correcta
                query_puntos = text("""
                        SELECT heading, distance, swathwidth, tilltype, sectionid, 
                               elevation, geom, isotime, vehiclspee
                        FROM siembra_surcado.detalles_lote_utm
                        WHERE lote_id = :lote_id 
                        ORDER BY isotime ASC
                    """)
                puntos = conn.execute(query_puntos, {
                    "lote_id": lote_id
                }).fetchall()
                
                # 4. Recorrer los puntos para crear segmentos punto a punto
                for i in range(len(puntos) - 1):
                    punto_actual = puntos[i]
                    punto_siguiente = puntos[i+1] # El segmento hereda datos de este
                    # Crear el segmento (LineString) entre punto i y punto i+1
                    # Usamos los datos del punto_siguiente (segundo punto)
                    insert_segmento = text("""
                            INSERT INTO siembra_surcado.segmentos_lineas (
                            data_lote_id, heading, distance, swathwidth, 
                            tilltype, sectionid, elevation, isotime, vehiclspee, geom
                            ) VALUES 
                                (
                                    :lote_id, :h, :d, :sw, :tt, :sid, :el, :iso, :vh,
                                    ST_MakeLine(:geom_a, :geom_b)
                                )
                            """)
                    conn.execute(insert_segmento, {
                        "lote_id": lote_id,
                        "h": punto_siguiente.heading,
                        "d": punto_siguiente.distance,
                        "sw": punto_siguiente.swathwidth,
                        "tt": punto_siguiente.tilltype,
                        "sid": punto_siguiente.sectionid,
                        "el": punto_siguiente.elevation,
                        "iso": punto_siguiente.isotime,
                        "vh": punto_siguiente.vehiclspee,
                        "geom_a": punto_actual.geom,
                        "geom_b": punto_siguiente.geom
                    })
                print(lote_id)
            # 5. Marcar lote como segmentado al terminar todas sus líneas
            conn.execute(text("""
                UPDATE siembra_surcado.data_lote 
                SET segmentos_creados = True 
                WHERE id = :lote_id
            """), {"lote_id": lote_id})
            print(f"Lote {lote_id} segmentado por intersección exitosamente.")
    except Exception as e:
        print(f"Error en el proceso: {e}")

In [72]:
# Ejecutar proceso
procesar_segmentacion_exacta()

--- Procesando Lote ID: 1 ---
1
Limpieza completada. Segmentos eliminados: 0
Limpieza de rumbos completada. Registros eliminados: 0
Lote 1 segmentado por intersección exitosamente.


In [74]:
def procesar_limpieza_de_segmentos():
    engine = obtener_engine()
    # 1. Recorrer los IDs de data_lote
    query_lotes = text("""
        SELECT id FROM siembra_surcado.data_lote 
        WHERE lineas_creadas = False
    """)
    try:
        with engine.begin() as conn:
            lotes = conn.execute(query_lotes).fetchall()
            for (lote_id,) in lotes:
                print(f"--- Limpieza Lote ID: {lote_id} ---")
                limpiar_segmentos_por_distancia_velocidad(lote_id)
                limpiar_segementos_por_rumbo(lote_id, 5)
            # 5. Marcar lote como limpiado
            conn.execute(text("""
                UPDATE siembra_surcado.data_lote 
                SET lineas_creadas = True 
                WHERE id = :lote_id
            """), {"lote_id": lote_id})
            print(f"Lote {lote_id} con limpieza de segmentos.")
    except Exception as e:
        print(f"Error en el proceso: {e}")

In [75]:
procesar_limpieza_de_segmentos()

--- Limpieza Lote ID: 1 ---
Limpieza completada. Segmentos eliminados: 2470
Limpieza de rumbos completada. Registros eliminados: 568
Lote 1 con limpieza de segmentos.


In [76]:
def calcular_desviacion_segmentos():
    engine = obtener_engine()
    
    # 1. Obtener los IDs de los lotes pendientes
    query_lotes = text("""
        SELECT id FROM siembra_surcado.data_lote 
        WHERE desviacion_calculada = False and segmentos_creados = True
    """)
    
    try:
        with engine.begin() as conn:
            lotes = conn.execute(query_lotes).fetchall()
            
            for (lote_id,) in lotes:
                print(f"--- Procesando Lote ID: {lote_id} ---")

                # Esta consulta usa RANK para identificar el vecino más cercano en tiempo
                # priorizando los menores (pasadas previas) y luego los mayores (pasadas posteriores)
                query_proceso = text("""
                    WITH calculo_base AS (
                        SELECT 
                            s1.id as seg_id,
                            s1.isotime as iso_orig,
                            ST_LineInterpolatePoint(s1.geom, 0.5) as centro,
                            ST_Azimuth(ST_StartPoint(s1.geom), ST_EndPoint(s1.geom)) as azimut,
                            ((s1.swathwidth + 1) * 2) as largo_sonda
                        FROM siembra_surcado.segmentos_lineas s1
                        WHERE s1.data_lote_id = :lote_id
                    ),
                    sonda_perpendicular AS (
                        SELECT 
                            seg_id,
                            iso_orig,
                            ST_MakeLine(
                                ST_Project(centro, largo_sonda/2, azimut + pi()/2),
                                ST_Project(centro, largo_sonda/2, azimut - pi()/2)
                            ) as linea_sonda,
                            centro
                        FROM calculo_base
                    ),
                    vecinos_detectados AS (
                        SELECT 
                            sp.seg_id,
                            sp.centro,
                            s2.geom as geom_vecino,
                            s2.isotime as iso_vecino,
                            -- Determinamos si es un vecino pasado (0) o futuro (1)
                            CASE WHEN s2.isotime < sp.iso_orig THEN 0 ELSE 1 END as orden_cronologico,
                            -- Calculamos la diferencia de tiempo absoluta para encontrar el más cercano
                            ABS(EXTRACT(EPOCH FROM (s2.isotime - sp.iso_orig))) as diferencia_tiempo
                        FROM sonda_perpendicular sp
                        JOIN siembra_surcado.segmentos_lineas s2 
                          ON ST_Intersects(sp.linea_sonda, s2.geom)
                        WHERE s2.data_lote_id = :lote_id
                          AND s2.id != sp.seg_id
                    ),
                    priorizacion AS (
                        SELECT 
                            seg_id,
                            centro,
                            geom_vecino,
                            ROW_NUMBER() OVER (
                                PARTITION BY seg_id 
                                ORDER BY 
                                    orden_cronologico ASC, -- Prioriza los pasados (0) sobre futuros (1)
                                    diferencia_tiempo ASC   -- Prioriza el más cercano en tiempo
                            ) as prioridad
                        FROM vecinos_detectados
                    )
                    UPDATE siembra_surcado.segmentos_lineas target
                    SET desviacion = ST_Distance(p.centro, p.geom_vecino)
                    FROM priorizacion p
                    WHERE target.id = p.seg_id 
                      AND p.prioridad = 1;
                """)
                print(f"Antes de ejecutar---")
                conn.execute(query_proceso, {"lote_id": lote_id})
                print(f"Depues de ejecutar---")

                # 3. Calcular Variación y Categoría manejando los casos sin vecino
                # Usamos COALESCE(desviacion, swathwidth) para que si es NULL, use el ancho teórico
                query_categorizacion = text("""
                    UPDATE siembra_surcado.segmentos_lineas
                    SET 
                        variacion = ABS(COALESCE(desviacion, swathwidth) - swathwidth),
                        categoria_variacion = CASE 
                            WHEN ABS(COALESCE(desviacion, swathwidth) - swathwidth) <= 0.05 THEN '1) <5 - OPTIMO'
                            WHEN ABS(COALESCE(desviacion, swathwidth) - swathwidth) <= 0.10 THEN '2) 5 a 10 - ACEPTABLE'
                            WHEN ABS(COALESCE(desviacion, swathwidth) - swathwidth) <= 0.15 THEN '3) 10 a 15 - RIESGO MODERADO'
                            ELSE '4) >15 - RIESGO ALTO'
                        END
                    WHERE data_lote_id = :lote_id;
                """)
                conn.execute(query_categorizacion, {"lote_id": lote_id})
                
                # 4. Actualización del estado del lote
                conn.execute(text("""
                    UPDATE siembra_surcado.data_lote 
                    SET desviacion_calculada = True
                    WHERE id = :lote_id
                """), {"lote_id": lote_id})
                print(f"  > Lote {lote_id} procesado íntegramente.")
                
    except Exception as e:
        print(f"Error en el proceso: {e}")

In [77]:
calcular_desviacion_segmentos()

--- Procesando Lote ID: 1 ---
Antes de ejecutar---
Depues de ejecutar---
  > Lote 1 procesado íntegramente.


In [7]:
def calcular_velocidad_segmentos():
    engine = obtener_engine()
    
    # 1. Obtener los IDs de los lotes pendientes
    query_lotes = text("""
        SELECT id FROM siembra_surcado.data_lote 
        WHERE velocidad_calculada = False AND segmentos_creados = True
    """)
    
    try:
        with engine.begin() as conn:
            lotes = conn.execute(query_lotes).fetchall()
            
            for (lote_id,) in lotes:
                print(f"--- Calculando Velocidades y Categorías Lote ID: {lote_id} ---")

                # 2. SQL con lógica de categorización añadida
                query_proceso_velocidad = text("""
                    WITH calculo_secuencial AS (
                        SELECT 
                            id,
                            isotime,
                            LAG(isotime) OVER (ORDER BY isotime) as iso_previo,
                            distance
                        FROM siembra_surcado.segmentos_lineas
                        WHERE data_lote_id = :lote_id
                    ),
                    metricas_calculadas AS (
                        SELECT 
                            id,
                            iso_previo,
                            CASE 
                                WHEN iso_previo IS NOT NULL AND (EXTRACT(EPOCH FROM (isotime - iso_previo))) > 0
                                THEN (distance / EXTRACT(EPOCH FROM (isotime - iso_previo))) * 3.6
                                ELSE 0 
                            END as velocidad_kmh
                        FROM calculo_secuencial
                    ),
                    categorizacion AS (
                        SELECT 
                            id,
                            iso_previo,
                            velocidad_kmh,
                            CASE 
                                WHEN velocidad_kmh <= 3.3 THEN '1) 3.3 km/h'
                                WHEN velocidad_kmh <= 4.3 THEN '2) 4.3 km/h'
                                WHEN velocidad_kmh <= 5.3 THEN '3) 5.3 km/h'
                                WHEN velocidad_kmh <= 6.3 THEN '4) 6.3 km/h'
                                WHEN velocidad_kmh <= 7.3 THEN '5) 7.3 km/h'
                                ELSE '6) >7.3 km/h'
                            END as cat_vel
                        FROM metricas_calculadas
                    )
                    UPDATE siembra_surcado.segmentos_lineas target
                    SET 
                        isotime_anterior = c.iso_previo,
                        velocidad = c.velocidad_kmh,
                        categoria_velocidad = c.cat_vel
                    FROM categorizacion c
                    WHERE target.id = c.id;
                """)
                
                conn.execute(query_proceso_velocidad, {"lote_id": lote_id})

                # 3. Marcar el lote como procesado
                conn.execute(text("""
                    UPDATE siembra_surcado.data_lote 
                    SET velocidad_calculada = True
                    WHERE id = :lote_id
                """), {"lote_id": lote_id})
                
                print(f"  > Velocidades y Categorías del Lote {lote_id} actualizadas.")

    except Exception as e:
        print(f"Error en el cálculo de velocidad: {e}")

In [8]:
calcular_velocidad_segmentos()

--- Calculando Velocidades y Categorías Lote ID: 17 ---
  > Velocidades y Categorías del Lote 17 actualizadas.
--- Calculando Velocidades y Categorías Lote ID: 18 ---
  > Velocidades y Categorías del Lote 18 actualizadas.


In [9]:
def clasificar_vehiclspee():
    engine = obtener_engine()
    
    # 1. Obtener IDs de lotes pendientes (puedes ajustar el nombre del flag si es necesario)
    query_lotes = text("""
        SELECT id FROM siembra_surcado.data_lote 
        WHERE vehiclspee_calculada = False AND segmentos_creados = True
    """)
    
    try:
        with engine.begin() as conn:
            lotes = conn.execute(query_lotes).fetchall()
            
            for (lote_id,) in lotes:
                print(f"--- Clasificando Categorías vehiclspee Lote ID: {lote_id} ---")

                # 2. SQL directo para clasificar el campo vehiclspee
                query_clasificacion = text("""
                    UPDATE siembra_surcado.segmentos_lineas
                    SET categoria_vehiclspee = CASE 
                        WHEN vehiclspee <= 3.3 THEN '1) 3.3 km/h'
                        WHEN vehiclspee <= 4.3 THEN '2) 4.3 km/h'
                        WHEN vehiclspee <= 5.3 THEN '3) 5.3 km/h'
                        WHEN vehiclspee <= 6.3 THEN '4) 6.3 km/h'
                        WHEN vehiclspee <= 7.3 THEN '5) 7.3 km/h'
                        ELSE '6) >7.3 km/h'
                    END
                    WHERE data_lote_id = :lote_id;
                """)
                
                conn.execute(query_clasificacion, {"lote_id": lote_id})

                # 3. Marcar el lote como procesado
                conn.execute(text("""
                    UPDATE siembra_surcado.data_lote 
                    SET vehiclspee_calculada = True
                    WHERE id = :lote_id
                """), {"lote_id": lote_id})
                
                print(f"  > Categorías de vehiclspee para el Lote {lote_id} actualizadas.")

    except Exception as e:
        print(f"Error en la clasificación de velocidad: {e}")

In [10]:
clasificar_vehiclspee()

--- Clasificando Categorías vehiclspee Lote ID: 17 ---
  > Categorías de vehiclspee para el Lote 17 actualizadas.
--- Clasificando Categorías vehiclspee Lote ID: 18 ---
  > Categorías de vehiclspee para el Lote 18 actualizadas.
